In [ ]:
!pip install openml

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn.functional as F
import time
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from unittest.mock import MagicMock, patch

In [ ]:
# Секция конфигурации и настройки датасетов
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen3-4B-Instruct-2507"
# Конфигурации датасетов
DATASET_CONFIGS = {

    "Bank": {
        "source": "openml", "openml_id": 1461,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Age','V2':'Job','V3':'Martial','V4':'Education',
            'V5':'Default','V6':'Balance','V7':'Housing','V8':'Loan',
            'V9':'Contact','V10':'Day of Week','V11':'Month','V12':'Duration',
            'V13':'Campaign','V14':'Pdays','V15':'Previous','V16':'Poutcome',
            'Class':'y'
        },
        "prompt_config": {
            "task": "Predict whether a bank client will subscribe",
            "labels": ["no", "yes"],
            "entity": "Client",
            "question": "Will this client subscribe to a term deposit?"
        },
    },

    "Blood": {
        "source": "openml", "openml_id": 1464,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Recency','V2':'Frequency','V3':'Monetary','V4':'Time',
            'Class':'Donated blood'
        },
        "prompt_config": {
            "task": "Predict whether a person donated blood",
            "labels": ["no", "yes"],
            "entity": "Donor",
            "question": "Did this person donate blood?"
        },
    },

    "California": {
        "source": "openml", "openml_id": 44090,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether house price is above median",
            "labels": ["no", "yes"],
            "entity": "House",
            "question": "Is this house price above median?"
        },
    },

    "Car": {
        "source": "openml", "openml_id": 40975,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict car evaluation",
            "labels": ["unacceptable", "acceptable", "good", "very good"],
            "entity": "Car",
            "question": "What is the evaluation of this car?"
        },
    },

    "Credit_G": {
        "source": "openml", "openml_id": 31,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Classify credit risk as good or bad",
            "labels": ["bad", "good"],
            "entity": "Client",
            "question": "Is this client a good credit risk?"
        },
    },

    "Diabetes": {
        "source": "kaggle",
        "kaggle_dataset": "uciml/pima-indians-diabetes-database",
        "kaggle_file": "diabetes.csv",
        "target_col": "Outcome",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has diabetes",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have diabetes?"
        },
    },

    "Heart": {
        "source": "kaggle",
        "kaggle_dataset": "fedesoriano/heart-failure-prediction",
        "kaggle_file": "heart.csv",
        "target_col": "HeartDisease",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has heart disease",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have heart disease?"
        },
    },

    "Income": {
        "source": "openml", "openml_id": 1590,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a person earns more than 50K a year",
            "labels": ["<=50K", ">50K"],
            "entity": "Person",
            "question": "Does this person earn more than 50K a year?"
        },
    },

    "Jungle": {
        "source": "openml", "openml_id": 41027,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict the endgame result of Jungle Chess",
            "labels": ["white_win", "draw", "black_win"],
            "entity": "Game Position",
            "question": "What is the game result?"
        },
    },
}

In [ ]:
# Загрузка данных
def load_openml(cfg):
    dataset = openml.datasets.get_dataset(cfg["openml_id"])
    target_col = dataset.default_target_attribute
    X, y, _, _ = dataset.get_data(dataset_format="dataframe", target=target_col)

    if cfg["feature_mappings"]:
        X = X.rename(columns=cfg["feature_mappings"])
        feature_names = list(cfg["feature_mappings"].values())[:-1]
        target_name = list(cfg["feature_mappings"].values())[-1]
    else:
        feature_names = X.columns.tolist()
        target_name = y.name

    X[target_name] = y

    le = LabelEncoder()
    X[target_name] = le.fit_transform(X[target_name].astype(str))
    return X, feature_names, target_name


def load_kaggle(cfg):
    import kagglehub
    path = kagglehub.dataset_download(cfg["kaggle_dataset"])
    df = pd.read_csv(f"{path}/{cfg['kaggle_file']}")
    target_name   = cfg["target_col"]
    feature_names = [c for c in df.columns if c != target_name]
    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name].astype(str))
    return df, feature_names, target_name

def load_dataset(cfg):
    """Загрузка тестовых выборок всех датасетов"""
    if cfg["source"] == "openml":
        df, feature_names, target_name = load_openml(cfg)
    elif cfg["source"] == "kaggle":
        df, feature_names, target_name = load_kaggle(cfg)
    else:
        raise ValueError(f"Unknown data source: {cfg['source']}")

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target_name])

    return train_df, test_df, feature_names, target_name

In [ ]:
# Пример загрузки датасета с openml
diabetes_cfg = DATASET_CONFIGS["Diabetes"]

train_df, test_df, feature_names, target_name = load_dataset(diabetes_cfg)

display(train_df.head())

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
353,1,90,62,12,43,27.2,0.580,24,0
711,5,126,78,27,22,29.6,0.439,40,0
373,2,105,58,40,94,34.9,0.225,25,0
46,1,146,56,0,0,29.7,0.564,29,0
682,0,95,64,39,105,44.6,0.366,22,0


In [ ]:
# Пример загрузки датасета с kaggle
bank_cfg = DATASET_CONFIGS["Bank"]

train_df, test_df, feature_names, target_name = load_dataset(bank_cfg)

display(train_df.head())

,Age,Job,Martial,Education,Default,Balance,Housing,Loan,Contact,Day of Week,Month,Duration,Campaign,Pdays,Previous,Poutcome,y
24001,36,technician,divorced,secondary,no,861.0,no,no,telephone,29,aug,140.0,2,-1.0,0.0,unknown,0
43409,24,student,single,secondary,no,4126.0,no,no,cellular,5,apr,907.0,4,185.0,7.0,failure,1
20669,44,technician,single,secondary,no,244.0,yes,no,cellular,12,aug,1735.0,4,-1.0,0.0,unknown,1
18810,48,unemployed,married,secondary,no,0.0,no,no,telephone,31,jul,35.0,11,-1.0,0.0,unknown,0
23130,38,technician,married,secondary,no,257.0,no,no,cellular,26,aug,57.0,10,-1.0,0.0,unknown,0


In [ ]:
def serialize_row(row, feature_names, format_type="template"):
    if format_type == "csv":
        return ", ".join([f"{f}: {row[f]}" for f in feature_names])
    elif format_type == "template":
        template_parts = []
        for feature in feature_names:
            val = row[feature]
            if isinstance(val, (int, np.integer)):
                template_parts.append(f"The value of {feature} is {val}.")
            elif isinstance(val, (float, np.floating)):
                template_parts.append(f"The value of {feature} is {val:.2f}.")
            else:
                template_parts.append(f"The category of {feature} is {val}.")
        return " ".join(template_parts)
    elif format_type == "html":
        headers = "".join([f"<th>{f}</th>" for f in feature_names])
        values = "".join([f"<td>{str(row[f]) if not isinstance(row[f], float) else f'{row[f]:.2f}'}</td>" for f in feature_names])
        return f"<table><tr>{headers}</tr><tr>{values}</tr></table>"
    elif format_type == "list":
        lines = []
        for feature in feature_names:
            val = row[feature]
            if isinstance(val, (float, np.floating)):
                lines.append(f"- {feature}: {val:.2f}")
            else:
                lines.append(f"- {feature}: {val}")
        return "\n".join(lines)

    else:
        raise ValueError("Unknown format type")

row = pd.Series({"Age": 32, "Job": "technician", "Balance": 1500.50})
feature_names = ["Age", "Job", "Balance"]

csv_out = serialize_row(row, feature_names, format_type="csv")
assert csv_out == "Age: 32, Job: technician, Balance: 1500.5"

markdown_out = serialize_row(row, feature_names, format_type="markdown")

# assert markdown_out == markdown_out_expected


json_out = serialize_row(row, feature_names, format_type="json")
assert json.loads(json_out)["Age"] == "32"

template_out = serialize_row(row, feature_names, format_type="template")
assert "The value of Age is 32." in template_out

html_out = serialize_row(row, feature_names, format_type="html")
assert "<td>32</td>" in html_out

list_out = serialize_row(row, feature_names, format_type="list")
assert list_out == "- Age: 32\n- Job: technician\n- Balance: 1500.50"


print(csv_out, "\n")
print(html_out, "\n")
print(template_out, "\n")
print(list_out, "\n")

Age: 32, Job: technician, Balance: 1500.5 

<table><tr><th>Age</th><th>Job</th><th>Balance</th></tr><tr><td>32</td><td>technician</td><td>1500.50</td></tr></table> 

The value of Age is 32. The category of Job is technician. The value of Balance is 1500.50. 

- Age: 32
- Job: technician
- Balance: 1500.50 



In [ ]:
def create_few_shot_prompt(target_row, train_df, feature_names, target_name, cfg, tokenizer, format_type, n_shots=16):
    num_classes = len(cfg['labels'])
    samples_per_class = max(1, n_shots // num_classes)

    few_shot_examples = pd.concat([train_df[train_df[target_name] == c].sample(n=samples_per_class, random_state=42, replace=True) for c in range(num_classes)])
    few_shot_examples = few_shot_examples.sample(frac=1, random_state=42)

    system_prompt = f"You are a classifier. {cfg['task']}. Answer with only one word: '{', '.join(cfg['labels'])}'."
    messages = [{"role": "system", "content": system_prompt}]

    for _, row in few_shot_examples.iterrows():
        text = serialize_row(row, feature_names, format_type)
        ans = cfg['labels'][int(row[target_name])]
        messages.append({"role": "user", "content": f"{cfg['entity']} info: \n{text}\n{cfg['question']}"})
        messages.append({"role": "assistant", "content": ans})

    client_text = serialize_row(target_row, feature_names, format_type)
    messages.append({"role": "user", "content": f"{cfg['entity']} info: \n{client_text}\n{cfg['question']}"})

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


train_df = pd.DataFrame({
        "Age": [20, 30, 40, 50],
        "target": [0, 1, 0, 1]
    })

target_row = pd.Series({"Age": 25})
cfg = {
    "task": "Predict something",
    "labels": ["yes", "no"],
    "entity": "User",
    "question": "Is it yes?"
}

mock_tokenizer = MagicMock()
mock_tokenizer.apply_chat_template.side_effect = lambda messages, **kwargs: str(messages)

prompt = create_few_shot_prompt(target_row, train_df, ["Age"], "target", cfg, mock_tokenizer, "csv", n_shots=2)

assert "You are a classifier." in prompt
assert "Is it yes?" in prompt
assert "Age: 25" in prompt

print(prompt)

[{'role': 'system', 'content': "You are a classifier. Predict something. Answer with only one word: 'yes, no'."}, {'role': 'user', 'content': 'User info: \nAge: 30\nIs it yes?'}, {'role': 'assistant', 'content': 'no'}, {'role': 'user', 'content': 'User info: \nAge: 20\nIs it yes?'}, {'role': 'assistant', 'content': 'yes'}, {'role': 'user', 'content': 'User info: \nAge: 25\nIs it yes?'}]


In [ ]:
def parse_prediction(response, labels):
    response = str(response).lower().strip().replace("'", "").replace('"', "").rstrip('.,!? \n')
    for i, label in enumerate(labels):
        if label.lower() in response: return i
    return 0

def predict_single(prompt, model, tokenizer, device, max_new_tokens=5):
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def predict_single_with_prob(prompt, feature_names, target_name, dataset_name, prompt_config, model, tokenizer, device, max_new_tokens=5):
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )

    # Декодирование текста
    generated_ids  = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для всех классов
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)

    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    # Возвращаем ответ и вероятности всех классов
    return response, probs.cpu().numpy()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    """Bootstrap метрики с доверительными интервалами"""
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            acc = accuracy_score(y_true_boot, y_pred_boot)
            f1 = f1_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)

            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
            else:
                auc = 0.0

            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map=device)

formats_to_test = ["csv", "template", "html", "list"]
results = []
# TEST_SAMPLE_SIZE = 100
for ds_name, cfg in DATASET_CONFIGS.items():
    train_df, test_df, feature_names, target_name = load_dataset(cfg)
    # test_subset = test_df.head(TEST_SAMPLE_SIZE)
    test_subset = test_df

    for fmt in tqdm(formats_to_test, desc=f"{ds_name}", leave=False):
        y_true, y_pred, y_prob_list = [], [], []
        t0 = time.time()
        prompt_config = cfg["prompt_config"]
        num_classes = len(prompt_config['labels'])

        for _, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc=f"{fmt}", leave=False):
            prompt = create_few_shot_prompt(row, train_df, feature_names, target_name, prompt_config, tokenizer, fmt, n_shots=16)
            response, probs = predict_single_with_prob(prompt, feature_names, target_name, ds_name, prompt_config, model, tokenizer, device)

            true_label = int(row[target_name])
            pred_label = np.argmax(probs)

            y_true.append(int(row[target_name]))
            y_pred.append(parse_prediction(response, prompt_config['labels']))

            if num_classes == 2:
                y_prob_list.append(probs[1])
            else:
                y_prob_list.append(probs)

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        y_prob_arr = np.array(y_prob_list)
        boot_metrics = bootstrap_metrics(y_true, y_pred, y_prob_arr, n_iter=1000)

        results.append({
            "Dataset": ds_name,
            "Format": fmt,
            "ROC-AUC": boot_metrics["ROC-AUC"],
            "F1_Macro": boot_metrics["F1"],
            "Accuracy": boot_metrics["Accuracy"],
            "Precision": boot_metrics["Precision"],
            "Recall": boot_metrics["Recall"],
            "Time_sec": round(time.time() - t0, 1)
        })

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Bank:   0%|          | 0/4 [00:00<?, ?it/s]

csv:   0%|          | 0/9043 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


...

Jungle:   0%|          | 0/4 [00:00<?, ?it/s]

csv:   0%|          | 0/8964 [00:00<?, ?it/s]

template:   0%|          | 0/8964 [00:00<?, ?it/s]

html:   0%|          | 0/8964 [00:00<?, ?it/s]

list:   0%|          | 0/8964 [00:00<?, ?it/s]

In [ ]:
print("Итоговые результаты")
df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("serialization_ablation_results.csv", index=False)

Итоговые результаты


,Dataset,Format,ROC-AUC,F1_Macro,Accuracy,Precision,Recall,Time_sec
0,Bank,csv,0.6468±0.0085,0.4794±0.0052,0.5637±0.0051,0.5489±0.0034,0.6180±0.0076,807.9
1,Bank,template,0.6570±0.0088,0.4944±0.0052,0.5905±0.0050,0.5504±0.0035,0.6205±0.0077,1121.3
2,Bank,html,0.6564±0.0097,0.5032±0.0051,0.6073±0.0048,0.5512±0.0036,0.6209±0.0078,1488.4
3,Bank,list,0.6265±0.0094,0.4946±0.0052,0.5970±0.0051,0.5464±0.0035,0.6103±0.0079,849.8
4,Blood,csv,0.6848±0.0439,0.4991±0.0393,0.5068±0.0394,0.5958±0.0312,0.6194±0.0373,10.2
5,Blood,template,0.6754±0.0476,0.5276±0.0394,0.5407±0.0394,0.6006±0.0309,0.6323±0.0387,12.1
6,Blood,html,0.6996±0.0439,0.5276±0.0390,0.5408±0.0390,0.6006±0.0305,0.6323±0.0381,13.9
7,Blood,list,0.6994±0.0449,0.5352±0.0393,0.5472±0.0393,0.6119±0.0306,0.6461±0.0378,11.1
8,California,csv,0.7439±0.0077,0.6777±0.0076,0.6777±0.0076,0.6778±0.0076,0.6777±0.0076,417.3
9,California,template,0.7311±0.0078,0.6626±0.0076,0.6639±0.0076,0.6665±0.0077,0.6640±0.0076,373.4


In [ ]:
import pandas as pd

def clean_and_float(value):
    if isinstance(value, str):
        return float(value.split('±')[0])
    return value

target_cols = ['ROC-AUC', 'F1_Macro', 'Accuracy', 'Precision', 'Recall', 'Time_sec']

for col in target_cols:
    if col in df_results.columns:
        df_results[col] = df_results[col].apply(clean_and_float)

summary_df = df_results.groupby('Format')[target_cols].mean()

final_rank = summary_df.sort_values(by='ROC-AUC', ascending=False)

print(final_rank.round(4))

          ROC-AUC  F1_Macro  Accuracy  Precision  Recall  Time_sec
Format                                                            
template   0.6640    0.5081    0.5436     0.5507  0.5598  444.0222
list       0.6617    0.5028    0.5383     0.5493  0.5603  385.3778
html       0.6610    0.4979    0.5285     0.5376  0.5562  560.8556
csv        0.6607    0.4999    0.5337     0.5484  0.5582  367.0000
